In [1]:
import pandas as pd
# from pathlib import Path

### Question 2. Understanding Docker networking and docker-compose
---
Given the following docker-compose.yaml, what is the hostname and port that pgadmin should use to connect to the postgres database?
```YAML
services:
  db:
    container_name: postgres
    image: postgres:17-alpine
    environment:
      POSTGRES_USER: 'postgres'
      POSTGRES_PASSWORD: 'postgres'
      POSTGRES_DB: 'ny_taxi'
    ports:
      - '5433:5432'
    volumes:
      - vol-pgdata:/var/lib/postgresql/data

  pgadmin:
    container_name: pgadmin
    image: dpage/pgadmin4:latest
    environment:
      PGADMIN_DEFAULT_EMAIL: "pgadmin@pgadmin.com"
      PGADMIN_DEFAULT_PASSWORD: "pgadmin"
    ports:
      - "8080:80"
    volumes:
      - vol-pgadmin_data:/var/lib/pgadmin

volumes:
  vol-pgdata:
    name: vol-pgdata
  vol-pgadmin_data:
    name: vol-pgadmin_data
```

Answer:
- db:5432
- postgres:5432

In [2]:
# # Get the current working directory
# current_dir = Path.cwd()

# # Safely append the file name (cross-platform: works on Linux, macOS, and Windows)
# green_tripdata_path = current_dir/"hw_data/green_tripdata_2025-11.parquet"
# taxi_zone_path = current_dir/"hw_data/taxi_zone_lookup.csv"

green_tripdata_path = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet'
taxi_zone_path = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv'

# Load data into DataFrames
green_tripdata_df = pd.read_parquet(green_tripdata_path)
taxi_zone_df = pd.read_csv(taxi_zone_path)

### Question 3. Counting short trips
---
For the trips in November 2025 (lpep_pickup_datetime between '2025-11-01' and '2025-12-01', exclusive of the upper bound), how many trips had a trip_distance of less than or equal to 1 mile?

In [3]:
# Filter rows for November 2025 (exclusive of December 1st)
november_data_df = green_tripdata_df[
    (green_tripdata_df['lpep_pickup_datetime'] >= '2025-11-01') & 
    (green_tripdata_df['lpep_pickup_datetime'] < '2025-12-01')
]

# Count the number of trips with a distance of 1 mile or less
short_trips_count = len(november_data_df[november_data_df['trip_distance'] <= 1])
print(f'Answer:\n\n{short_trips_count} trips had a trip_distance of less than or equal to 1 mile')

# # An alternative way using sum()
# short_trips_count = (november_data_df['trip_distance'] <= 1).sum()
# short_trips_count

Answer:

8007 trips had a trip_distance of less than or equal to 1 mile


### Question 4. Longest trip for each day

---

Which was the pick up day with the longest trip distance? Only consider trips with trip_distance less than 100 miles (to exclude data errors).

Use the pick up time for your calculations.

- 2025-11-14
- 2025-11-20
- 2025-11-23
- 2025-11-25

**Date Filtering Strategy & Performance Analysis:**

To filter and group by date, three approaches can be applied:
1. **`.dt.date`** — Converts timestamps to Python `datetime.date` objects (`object` dtype). It requires strict type-matching, meaning a list of raw strings inside `.isin()` will not match these objects.
2. **`.dt.strftime('%Y-%m-%d')`** — Converts timestamps into pure strings (`object` or `string` dtype). It allows using a simple list of raw strings inside `.isin()`, but introduces string manipulation overhead.
3. **`.dt.normalize()`** — Clamps the time to midnight while preserving the native, highly optimized `datetime64[us]` numeric format.

**Decision:** Option 3 (`.dt.normalize()`) combined with `pd.to_datetime()` for the mask is selected. This ensures maximum execution speed and vectorization safety on large-scale datasets.

In [4]:
# Filter rows for trips shorter than 100 miles
short_trips_df = green_tripdata_df[green_tripdata_df['trip_distance'] < 100].copy()

# Normalize timestamp to midnight for precise daily grouping
short_trips_df['lpep_pickup_date'] = short_trips_df['lpep_pickup_datetime'].dt.normalize()

# Define targeted dates and filter using explicit datetime parsing
dates_to_filter = pd.to_datetime(['2025-11-14', '2025-11-20', '2025-11-23', '2025-11-25'])
short_trips_df_filtered = short_trips_df[short_trips_df['lpep_pickup_date'].isin(dates_to_filter)]


In [5]:
# Find the date with the maximum trip distance directly
best_day = short_trips_df_filtered.groupby('lpep_pickup_date')['trip_distance'].max().idxmax()

# Format the result for output
date_str = best_day.strftime('%Y-%m-%d')
print(f'Answer:\n\nThe pick up day with the longest trip distance is {date_str}')

Answer:

The pick up day with the longest trip distance is 2025-11-14


### Question 5. Biggest pickup zone
---
Which was the pickup zone with the largest total_amount (sum of all trips) on November 18th, 2025?

- East Harlem North
- East Harlem South
- Morningside Heights
- Forest Hills

In [6]:
# Filter rows directly by date range
november_eighteenth_df = green_tripdata_df[
    (green_tripdata_df['lpep_pickup_datetime'] >= '2025-11-18') & 
    (green_tripdata_df['lpep_pickup_datetime'] < '2025-11-19')
]

# Find the location id with the largest total_amount directly
indx_to_find = november_eighteenth_df.groupby('PULocationID')['total_amount'].sum().idxmax()

# Extract the scalar zone string via single-pass row/column alignment to avoid double-bracket slicing
pickup_zone = taxi_zone_df.loc[taxi_zone_df['LocationID'] == indx_to_find, 'Zone'].values[0]

# Format the result for output
print(f'Answer:\n\nThe pickup zone with the largest total_amount on November 18th, 2025 is: {pickup_zone}')

Answer:

The pickup zone with the largest total_amount on November 18th, 2025 is: East Harlem North


### Question 6. Largest tip
---
For the passengers picked up in the zone named "East Harlem North" in November 2025, which was the drop off zone that had the largest tip?

**Note:** it's tip , not trip. We need the name of the zone, not the ID.

- JFK Airport
- Yorkville West
- East Harlem North
- LaGuardia Airport

In [7]:
# Retrieve the location ID for the specified pickup zone
pickup_zone_name = 'East Harlem North'
pickup_zone_id = taxi_zone_df.loc[taxi_zone_df['Zone'] == pickup_zone_name, 'LocationID'].values[0]

# Find the drop-off zone with the largest tip amount for trips originating from the specified pickup zone
largest_tip_zone_id = november_data_df[november_data_df['PULocationID'] == pickup_zone_id].groupby('DOLocationID')['tip_amount'].max().idxmax()

# Map the resulting drop-off location ID back to its textual zone name
target_zone_name = taxi_zone_df.loc[taxi_zone_df['LocationID'] == largest_tip_zone_id, 'Zone'].values[0]

# Format the result for output
print(f'Answer:\n\nThe drop off zone is: {target_zone_name}')

Answer:

The drop off zone is: Yorkville West
